# 05 — Drift Detection & Data Governance
**Demand Signal Feature Store — Capstone Project**

This notebook covers:
1. **PSI** (Population Stability Index) — detect feature-level drift between reference and current periods
2. **CSI** (Characteristic Stability Index) — drill into which bins shifted for drifting features
3. **Great Expectations** — data quality validation on the feature store
4. **Freshness SLA** monitoring
5. **Lineage** audit

The key insight: drift in LLM-extracted features (sentiment, risk_mention_rate) signals a change in **how suppliers write** — an early warning that appears *before* the tabular metrics shift.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.drift import calculate_psi_for_features, calculate_csi
from src.feature_store import check_freshness_sla, get_lineage_report

sns.set_style("whitegrid")

## 1. Load Feature Store & Define Periods

In [ ]:
feature_store = pd.read_csv("../data/feature_store.csv")
print(f"Feature store: {feature_store.shape}")

all_months = sorted(feature_store["year_month"].unique())
mid = len(all_months) // 2

reference_months = all_months[:mid]
current_months = all_months[mid:]

reference_df = feature_store[feature_store["year_month"].isin(reference_months)]
current_df = feature_store[feature_store["year_month"].isin(current_months)]

print(f"Reference period: {reference_months[0]} to {reference_months[-1]} ({len(reference_df)} rows)")
print(f"Current period:   {current_months[0]} to {current_months[-1]} ({len(current_df)} rows)")

## 2. PSI — Feature Drift Detection

In [ ]:
DRIFT_FEATURES = [
    # ERP features
    "avg_lead_time", "avg_delay", "on_time_rate", "fulfillment_rate", "avg_unit_cost",
    # LLM-extracted features
    "risk_mention_rate", "avg_delay_signal", "avg_sentiment", "min_sentiment",
    "capacity_is_constrained",
]

psi_results = calculate_psi_for_features(reference_df, current_df, DRIFT_FEATURES)
print("=== PSI Results ===")
print(psi_results.to_string(index=False))

In [ ]:
# Visualize PSI
psi_plot = psi_results.dropna(subset=["psi"]).sort_values("psi", ascending=True)

colors = []
for _, row in psi_plot.iterrows():
    if row["status"] == "stable":
        colors.append("#5b9f5b")
    elif row["status"] == "moderate_drift":
        colors.append("#d4a44a")
    else:
        colors.append("#d4726a")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(psi_plot["feature"], psi_plot["psi"], color=colors)
ax.axvline(x=0.1, color="orange", linestyle="--", alpha=0.7, label="Moderate threshold (0.1)")
ax.axvline(x=0.2, color="red", linestyle="--", alpha=0.7, label="Significant threshold (0.2)")
ax.set_xlabel("PSI")
ax.set_title("Feature Drift Detection (PSI)\nReference vs Current Period")
ax.legend()
plt.tight_layout()
plt.savefig("../data/psi_drift.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. CSI — Drill Into Drifting Features

In [ ]:
# Pick the top drifting feature for CSI analysis
drifting = psi_results[psi_results["status"] != "stable"].sort_values("psi", ascending=False)

if len(drifting) > 0:
    top_feature = drifting.iloc[0]["feature"]
    print(f"CSI breakdown for most drifting feature: {top_feature}")
    csi = calculate_csi(reference_df, current_df, top_feature)
    print(csi.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(csi))
    ax.bar(x, csi["reference_pct"], alpha=0.6, label="Reference", color="#6a8fc4")
    ax.bar(x, csi["current_pct"], alpha=0.6, label="Current", color="#d4726a")
    ax.set_xticks(x)
    ax.set_xticklabels(csi["bin"], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Proportion")
    ax.set_title(f"CSI Breakdown: {top_feature}")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../data/csi_breakdown.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No significant drift detected — all features stable.")

## 4. Great Expectations — Data Quality Validation

In [ ]:
import great_expectations as gx

context = gx.get_context()

ds = context.sources.add_or_update_pandas(name="feature_store_source")
asset = ds.add_dataframe_asset(name="feature_store", dataframe=feature_store)
batch_request = asset.build_batch_request()

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="feature_store_quality",
    create_expectation_suite_with_name="feature_store_quality",
)

In [ ]:
# Define expectations
validator.expect_column_to_exist("supplier_id")
validator.expect_column_to_exist("year_month")

validator.expect_column_values_to_not_be_null("supplier_id")
validator.expect_column_values_to_not_be_null("year_month")
validator.expect_column_values_to_not_be_null("total_orders")
validator.expect_column_values_to_not_be_null("avg_lead_time")

validator.expect_column_values_to_be_between("on_time_rate", min_value=0, max_value=1)
validator.expect_column_values_to_be_between("fulfillment_rate", min_value=0, max_value=1.5)
validator.expect_column_values_to_be_between("avg_lead_time", min_value=0, max_value=120)

# LLM feature quality
validator.expect_column_values_to_be_between("risk_mention_rate", min_value=0, max_value=1)
validator.expect_column_values_to_be_between("avg_sentiment", min_value=-1, max_value=1)

validator.expect_compound_columns_to_be_unique(["supplier_id", "year_month"])

print("Expectations defined.")

In [ ]:
# Run validation
results = validator.validate()

print(f"\n=== Data Quality Validation ===")
print(f"Success: {results.success}")
print(f"Expectations evaluated: {results.statistics['evaluated_expectations']}")
print(f"Successful: {results.statistics['successful_expectations']}")
print(f"Failed: {results.statistics['unsuccessful_expectations']}")

if not results.success:
    print("\nFailed expectations:")
    for r in results.results:
        if not r.success:
            print(f"  - {r.expectation_config.expectation_type}: {r.result}")

## 5. Freshness SLA Monitoring

In [ ]:
from config.settings import FRESHNESS_SLA_HOURS

sla_df = check_freshness_sla(feature_store)

print(f"=== Freshness SLA ({FRESHNESS_SLA_HOURS}h) ===")
print(f"Records meeting SLA: {sla_df['freshness_sla_met'].sum()}/{len(sla_df)} ({sla_df['freshness_sla_met'].mean():.1%})")

if "hours_since_update" in sla_df.columns:
    print(f"\nAge distribution (hours since last update):")
    print(sla_df["hours_since_update"].describe().round(1))

## 6. Lineage Audit

In [ ]:
lineage = get_lineage_report(feature_store)

print("=== Lineage Audit ===")
print(f"Total rows in feature store: {len(lineage)}")
print(f"Rows with ERP features: {lineage['has_erp_features'].sum()} ({lineage['has_erp_features'].mean():.1%})")
print(f"Rows with LLM features: {lineage['has_llm_features'].sum()} ({lineage['has_llm_features'].mean():.1%})")

# Coverage by supplier
coverage = lineage.groupby("supplier_id").agg(
    months=("year_month", "count"),
    llm_coverage=("has_llm_features", "mean"),
).round(2)
print("\nLLM feature coverage by supplier:")
print(coverage.to_string())

## 7. Summary Dashboard

In [ ]:
print("=" * 60)
print("DATA GOVERNANCE SUMMARY")
print("=" * 60)
print(f"\n{'Metric':<35} {'Status':<15} {'Value'}")
print("-" * 60)

# Data quality
dq_status = "PASS" if results.success else "FAIL"
print(f"{'Data Quality (GE)':<35} {dq_status:<15} {results.statistics['successful_expectations']}/{results.statistics['evaluated_expectations']} passed")

# Freshness
fr_pct = sla_df["freshness_sla_met"].mean()
fr_status = "PASS" if fr_pct >= 0.95 else "WARN" if fr_pct >= 0.80 else "FAIL"
print(f"{'Freshness SLA ({FRESHNESS_SLA_HOURS}h)':<35} {fr_status:<15} {fr_pct:.1%}")

# Drift
n_drifting = len(psi_results[psi_results["status"] != "stable"])
drift_status = "PASS" if n_drifting == 0 else "WARN" if n_drifting <= 2 else "ALERT"
print(f"{'Feature Drift (PSI)':<35} {drift_status:<15} {n_drifting}/{len(DRIFT_FEATURES)} features drifting")

# Lineage
llm_cov = lineage["has_llm_features"].mean()
lin_status = "PASS" if llm_cov >= 0.90 else "WARN" if llm_cov >= 0.70 else "FAIL"
print(f"{'Lineage Coverage':<35} {lin_status:<15} {llm_cov:.1%}")

print("=" * 60)